In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

## Data Analysis

In [ ]:
x_train = pd.read_csv('x_train.csv')
y_train = pd.read_csv('y_train.csv')
train = pd.concat([x_train, y_train], axis=1)
test = pd.read_csv('x_test.csv')
train.head()

In [ ]:
print(x_train.shape[0])
print(x_train.shape[1])

In [ ]:
train=train.dropna()
test=test.dropna()

In [ ]:
colstrain = [f'RET_{i}' for i in range(1, 21)] 
train['Ma20'] = train[colstrain].mean(axis=1)

coltest=[f'RET_{i}' for i in range(1,21)]
test['Ma20']= test[coltest].mean(axis=1)

train['Ma5']=train[[f'RET_{i}' for i in range(1,6)]].mean(axis=1)

train['V5']=train[[f'RET_{i}' for i in range(1,6)]].std(axis=1)

train['V20']=train[[f'RET_{i}' for i in range(1,21)]].std(axis=1)

train['RET'] = LabelEncoder().fit_transform(train['RET'])

train.head()

In [ ]:
print(train['SECTOR'].value_counts())

In [ ]:
print(train['INDUSTRY'].value_counts())

In [ ]:
print(train['INDUSTRY_GROUP'].value_counts())

In [ ]:
train['Sector_Ma20'] = train.groupby('SECTOR')['Ma20'].transform(lambda x: x.rolling(window=20, min_periods=1).mean())
train['Ind_Ma20'] = train.groupby('INDUSTRY')['Ma20'].transform(lambda x: x.rolling(window=20, min_periods=1).mean())
train['IndGroup_Ma20'] = train.groupby('INDUSTRY_GROUP')['Ma20'].transform(lambda x: x.rolling(window=20, min_periods=1).mean())
train['Ind_Ma5'] = train.groupby('INDUSTRY')['V20'].transform(lambda x: x.rolling(window=5, min_periods=1).mean())

train.head()

In [ ]:
train = train.drop(['ID','STOCK', 'DATE','INDUSTRY','INDUSTRY_GROUP','SECTOR','SUB_INDUSTRY'], axis = 1) 
test=test.drop(['ID','STOCK', 'DATE','INDUSTRY','INDUSTRY_GROUP','SECTOR','SUB_INDUSTRY'],axis=1)

In [ ]:
train.head()

In [ ]:
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold,cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,roc_curve,classification_report,auc
from sklearn import tree,metrics

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold,cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,roc_curve,classification_report,auc
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn import tree
import seaborn as sns
import numpy as np
import pandas as pd
#import xgboost as xgb
#from xgboost import XGBClassifier

## Evaluation Functions

In [ ]:
def modelfit(model,X,Y,features, performCV=True,roc=False, printFeatureImportance=False):
    
    #Fitting the model on the data_set
    model.fit(X[features],Y)
        
    #Predict training set:
    predictions = model.predict(X[features])
    predprob = model.predict_proba(X[features])[:,1]
    
    # Create and print confusion matrix    
    cfm = confusion_matrix(Y,predictions)
    print("\nModel Confusion matrix")
    print(cfm)
    
    #Print model report:
    print("\nModel Report")
    print("Accuracy : %.4g" % metrics.accuracy_score(Y.values, predictions))
    
    #Perform cross-validation: evaluate using 10-fold cross validation 
    #kfold = StratifiedKFold(n_splits=10, shuffle=True)
    kfold = 2
    if performCV:
        evaluation(model,X[features],Y,kfold)
    if roc: 
        compute_roc(Y, predictions, plot=True)
          
    #Print Feature Importance:
    if printFeatureImportance:
        feature_importance(model,features,selection=False) 

In [ ]:
def evaluation(model,X,Y,kfold):
    # Cross Validation to test and anticipate overfitting problem
    scores1 = cross_val_score(model, X, Y, cv=kfold, scoring='accuracy')
    scores2 = cross_val_score(model, X, Y, cv=kfold, scoring='precision')
    scores3 = cross_val_score(model, X, Y, cv=kfold, scoring='recall')
    # The mean score and standard deviation of the score estimate
    print("Cross Validation Accuracy: %0.5f (+/- %0.2f)" % (scores1.mean(), scores1.std()))
    print("Cross Validation Precision: %0.5f (+/- %0.2f)" % (scores2.mean(), scores2.std()))
    print("Cross Validation Recall: %0.5f (+/- %0.2f)" % (scores3.mean(), scores3.std()))
    return  

In [ ]:
def compute_roc(Y, y_pred, plot=True):
    fpr = dict()
    tpr = dict()
    auc_score = dict()
    fpr, tpr, _ = roc_curve(Y, y_pred)
    auc_score = auc(fpr, tpr)
    if plot:
        plt.figure(figsize=(7, 6))
        plt.plot(fpr, tpr, color='blue',
                 label='ROC curve (area = %0.2f)' % auc_score)
        plt.plot([0, 1], [0, 1], color='navy', lw=3, linestyle='--')
        plt.legend(loc="upper right")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.0])
        plt.title("ROC Curve")
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver operating characteristic')
        plt.show()
    return fpr, tpr, auc_score

In [ ]:
def feature_importance(model,features,selection=False): 

    importance=pd.DataFrame(model.feature_importances_ )    
    data=pd.DataFrame(data={'Values': model.feature_importances_ ,'Features':features})
    data=data.sort_values(by="Values",ascending=False)

    plt.figure(figsize=(12, 8))
    plt.bar(data["Features"], data["Values"], color="blue")
    plt.xticks(rotation=90)
    plt.xlabel("Features")
    plt.ylabel("Values")
    plt.title("Feature Importance in Descending Order")
    plt.tight_layout()

    if selection:
        threshold=0.02
        filt=data[data["Values"] > threshold]["Features"].tolist()
        print(filt)
    
    #plt.show()
    return plt

### Decision Tree approach

In [ ]:
x_traina, x_testa, y_traina, y_testa = train_test_split(train.loc[:, train.columns != 'RET'], train.RET, test_size=0.25, random_state =0)

features=[x for x in train.columns if x not in ['RET']]
model = tree.DecisionTreeClassifier()
modelfit(model,x_traina, y_traina, features, performCV=False)
print("Accuracy on test set :{:.3f} ".format(model.score(x_testa, y_testa)))

In [ ]:
print('Decision tree with Classifier')
params={'max_depth': np.arange(5, 10),'criterion':['gini','entropy']}
tree_estimator = tree.DecisionTreeClassifier()

grid_tree = GridSearchCV(tree_estimator, params, cv=3, scoring="accuracy",
                         n_jobs=1,
                         verbose=True)

grid_tree.fit(x_traina, y_traina)
best_est = grid_tree.best_estimator_
print(best_est)
print(grid_tree.best_score_)


# summarize results
print("Best: %f using %s" % (grid_tree.best_score_, grid_tree.best_params_))
means = grid_tree.cv_results_['mean_test_score']
stds = grid_tree.cv_results_['std_test_score']
params = grid_tree.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))


In [ ]:
model = tree.DecisionTreeClassifier(max_depth = 6,criterion='gini')
modelfit(model,x_traina, y_traina,features,printFeatureImportance=False)

In [ ]:
feature_importance(model,features,selection=True)

In [ ]:
New_Features=['V20', 'RET_1', 'Ind_Ma20', 'VOLUME_1', 'RET_7', 'RET_17', 'VOLUME_7', 'VOLUME_18', 'IndGroup_Ma20', 'RET_4', 'RET_8', 'VOLUME_12', 'VOLUME_3', 'RET_20', 'RET_5', 'RET_3', 'VOLUME_13', 'RET_11', 'RET_16']

In [ ]:
print(New_Features)

In [ ]:
x_train_sl, x_test_sl, y_train_sl, y_test_sl = train_test_split(train.loc[:, train[New_Features].columns], train.RET, random_state =0)
model = tree.DecisionTreeClassifier(max_depth = 5,criterion='gini')
modelfit(model,x_train_sl, y_train_sl,New_Features,printFeatureImportance=True)

In [ ]:
ni